# PeakWeights: Experimental Results Generation

This notebook generates all experimental results for the PeakWeights paper.

## Setup Instructions
1. **Runtime > Change runtime type**
2. **Select GPU: A100 (recommended) or H100**
3. **Run all cells in order**

Results will be automatically downloaded at the end.

## 1. GPU Setup & Verification

In [ ]:
# Check GPU first
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                       capture_output=True, text=True)
print("GPU Information:")
print(result.stdout)

# Verify we have A100 or H100
gpu_name = result.stdout.strip().lower()
if 'a100' in gpu_name:
    print("\n✅ A100 GPU detected - Perfect for experiments!")
    GPU_TYPE = 'A100'
elif 'h100' in gpu_name:
    print("\n✅ H100 GPU detected - Excellent for experiments!")
    GPU_TYPE = 'H100'
elif 'v100' in gpu_name:
    print("\n⚠️ V100 GPU detected - Will run smaller models only")
    GPU_TYPE = 'V100'
elif 't4' in gpu_name:
    print("\n⚠️ T4 GPU detected - Will run smallest models only")
    print("   Consider switching to A100 runtime for full experiments")
    GPU_TYPE = 'T4'
else:
    print(f"\n⚠️ Unknown GPU: {gpu_name}")
    GPU_TYPE = 'Unknown'

## 2. Install Dependencies & PeakWeights

In [ ]:
# Install core dependencies
!pip install -q torch transformers accelerate
!pip install -U -q bitsandbytes  # Must be latest version for 4-bit quantization
!pip install -q datasets sentencepiece protobuf scipy
!pip install -q huggingface_hub matplotlib seaborn

# Install PeakWeights from GitHub
!pip install -q git+https://github.com/Kalmantic/peakweights.git

# Verify bitsandbytes installation
import bitsandbytes as bnb
print(f"\n✅ All dependencies installed!")
print(f"   bitsandbytes version: {bnb.__version__}")

In [ ]:
# Optional: Login to HuggingFace (only needed for gated models like Llama, Gemma)
# The default model list uses non-gated models, so this step is OPTIONAL

from huggingface_hub import login, whoami

try:
    user_info = whoami()
    print(f"Already logged in as: {user_info['name']}")
except:
    print("Not logged in to HuggingFace.")
    print("This is OPTIONAL - the default models don't require login.")
    print("If you want to use gated models (Llama, Gemma), uncomment and run:")
    print("  # login()")
    
# Uncomment below to login for gated models:
# login()

In [ ]:
import torch
import json
import time
import heapq
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Check GPU memory
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {gpu_memory_gb:.1f} GB")

## 3. Import PeakWeights from Package

The core algorithm is imported from the installed package - no code replication!

In [ ]:
# Import PeakWeights from installed package
import peakweights
from peakweights import PeakWeightsFinder, find as peakweights_find, CriticalWeight

print(f"PeakWeights package imported successfully!")
print(f"  - PeakWeightsFinder: For full analysis with model loading")
print(f"  - peakweights_find(): Quick API for finding critical weights")
print(f"  - CriticalWeight: Data class for critical weight results")

# Helper class to extend PeakWeightsFinder for collecting all scores (needed for power law plots)
class PeakWeightsAnalyzer:
    """Wrapper around PeakWeightsFinder that collects all scores for power law analysis."""
    
    def __init__(self, model, tokenizer, device='cuda'):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.all_scores = []  # For power law analysis
        self.layer_scores = {}
        self.activations = {}
        self.hooks = []
        
    def _register_hooks(self):
        """Register forward hooks to capture activations."""
        def make_hook(name):
            def hook(module, input, output):
                if isinstance(input, tuple) and len(input) > 0:
                    inp = input[0]
                    if isinstance(inp, torch.Tensor):
                        with torch.no_grad():
                            max_act = inp.abs().max(dim=0).values
                            if len(max_act.shape) > 1:
                                max_act = max_act.max(dim=0).values
                            self.activations[name] = max_act.clone()
            return hook
        
        for name, module in self.model.named_modules():
            if isinstance(module, torch.nn.Linear):
                hook = module.register_forward_hook(make_hook(name))
                self.hooks.append(hook)
    
    def _remove_hooks(self):
        for hook in self.hooks:
            hook.remove()
        self.hooks = []
    
    def find_critical_weights(self, top_k: int = 100, num_tokens: int = 128, 
                              collect_all_scores: bool = True) -> List[CriticalWeight]:
        """Find critical weights using PeakWeights algorithm: score = |weight| × |max_activation|"""
        print(f"Running PeakWeights analysis (top_k={top_k}, tokens={num_tokens})...")
        
        self.activations = {}
        self.all_scores = []
        self._register_hooks()
        
        # Run forward pass with synthetic tokens
        start_time = time.time()
        vocab_size = self.model.config.vocab_size
        input_ids = torch.randint(0, vocab_size, (1, num_tokens), device=self.device)
        
        with torch.no_grad():
            self.model(input_ids, use_cache=False)
        forward_time = time.time() - start_time
        print(f"  Forward pass: {forward_time:.2f}s")
        
        self._remove_hooks()
        
        # Score weights using PeakWeights formula: |weight| × |max_activation|
        heap = []
        
        start_time = time.time()
        for name, module in self.model.named_modules():
            if isinstance(module, torch.nn.Linear) and name in self.activations:
                weight = module.weight.data
                max_act = self.activations[name]
                
                if max_act.shape[0] != weight.shape[1]:
                    continue
                
                # PeakWeights core formula
                scores = weight.abs() * max_act.unsqueeze(0)
                
                # Collect for power law analysis
                if collect_all_scores:
                    flat_scores = scores.flatten().cpu().numpy()
                    if len(flat_scores) > 100000:
                        indices = np.random.choice(len(flat_scores), 100000, replace=False)
                        flat_scores = flat_scores[indices]
                    self.all_scores.extend(flat_scores.tolist())
                
                # Store layer score
                self.layer_scores[name] = scores.sum().item()
                
                # Find top-k using heap
                flat_scores_tensor = scores.flatten()
                layer_k = min(top_k, flat_scores_tensor.numel())
                top_vals, top_indices = torch.topk(flat_scores_tensor, layer_k)
                
                for val, idx in zip(top_vals.tolist(), top_indices.tolist()):
                    i = idx // weight.shape[1]
                    j = idx % weight.shape[1]
                    w_val = weight[i, j].item()
                    m_act = max_act[j].item()
                    
                    if len(heap) < top_k:
                        heapq.heappush(heap, (val, name, i, j, w_val, m_act))
                    elif val > heap[0][0]:
                        heapq.heapreplace(heap, (val, name, i, j, w_val, m_act))
        
        scan_time = time.time() - start_time
        print(f"  Weight scan: {scan_time:.2f}s")
        
        # Convert to CriticalWeight objects
        results = sorted(heap, key=lambda x: -x[0])
        critical_weights = [
            CriticalWeight(
                rank=i+1,
                score=score,
                module=name,
                param='weight',
                flat_index=row * 1000 + col,  # Approximate
                row=row,
                col=col,
                value=w_val
            )
            for i, (score, name, row, col, w_val, m_act) in enumerate(results)
        ]
        
        print(f"  Found {len(critical_weights)} critical weights")
        print(f"  Collected {len(self.all_scores)} scores for distribution analysis")
        return critical_weights

print("PeakWeightsAnalyzer wrapper loaded (extends package for power law visualization)")

## 4. Power Law Analysis & Visualization

In [ ]:
def plot_power_law_distribution(scores: List[float], model_name: str, save_path: str = None):
    """
    Plot the weight importance score distribution and fit power law.
    
    Power law: P(x) ∝ x^(-α)
    In log-log space: log(P) = -α * log(x) + c
    """
    scores = np.array(scores)
    scores = scores[scores > 0]  # Remove zeros
    scores = np.sort(scores)[::-1]  # Sort descending
    
    # Create rank array
    ranks = np.arange(1, len(scores) + 1)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Plot 1: Log-log rank vs score (Zipf's law style)
    ax1 = axes[0]
    ax1.loglog(ranks, scores, 'b.', alpha=0.3, markersize=1)
    
    # Fit power law to top portion
    fit_range = min(10000, len(scores) // 10)
    log_ranks = np.log10(ranks[:fit_range])
    log_scores = np.log10(scores[:fit_range])
    slope, intercept, r_value, p_value, std_err = stats.linregress(log_ranks, log_scores)
    
    fit_line = 10**(intercept + slope * np.log10(ranks))
    ax1.loglog(ranks, fit_line, 'r-', linewidth=2, label=f'Power law fit (α={-slope:.2f})')
    
    ax1.set_xlabel('Rank', fontsize=12)
    ax1.set_ylabel('Importance Score', fontsize=12)
    ax1.set_title(f'{model_name}\nRank vs Score (log-log)', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Score distribution (histogram)
    ax2 = axes[1]
    log_scores_all = np.log10(scores + 1e-10)
    ax2.hist(log_scores_all, bins=100, density=True, alpha=0.7, color='steelblue')
    ax2.set_xlabel('log₁₀(Score)', fontsize=12)
    ax2.set_ylabel('Density', fontsize=12)
    ax2.set_title('Score Distribution (log scale)', fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Cumulative importance
    ax3 = axes[2]
    cumsum = np.cumsum(scores) / np.sum(scores) * 100
    ax3.semilogx(ranks, cumsum, 'g-', linewidth=2)
    
    # Mark key points
    for k in [10, 100, 1000]:
        if k < len(cumsum):
            ax3.axvline(x=k, color='red', linestyle='--', alpha=0.5)
            ax3.annotate(f'Top-{k}: {cumsum[k-1]:.1f}%', 
                        xy=(k, cumsum[k-1]), 
                        xytext=(k*2, cumsum[k-1]-5),
                        fontsize=9)
    
    ax3.set_xlabel('Number of Weights (log scale)', fontsize=12)
    ax3.set_ylabel('Cumulative Importance (%)', fontsize=12)
    ax3.set_title('Cumulative Importance Distribution', fontsize=12)
    ax3.set_ylim(0, 100)
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  Plot saved to {save_path}")
    
    plt.show()
    
    # Return statistics
    return {
        'power_law_exponent': -slope,
        'r_squared': r_value**2,
        'top_10_cumulative': cumsum[9] if len(cumsum) > 9 else None,
        'top_100_cumulative': cumsum[99] if len(cumsum) > 99 else None,
        'top_1000_cumulative': cumsum[999] if len(cumsum) > 999 else None,
    }

def plot_critical_weight_locations(critical_weights: List, model_name: str, save_path: str = None):
    """Visualize where critical weights are located in the model."""
    
    # Categorize weights (use .module attribute from CriticalWeight)
    categories = {'attention': [], 'mlp': [], 'embed': [], 'output': [], 'gate': [], 'other': []}
    
    for cw in critical_weights:
        module = cw.module.lower()
        if 'attn' in module or 'attention' in module:
            categories['attention'].append(cw)
        elif 'gate' in module or 'router' in module:
            categories['gate'].append(cw)
        elif 'mlp' in module or 'ffn' in module:
            categories['mlp'].append(cw)
        elif 'embed' in module:
            categories['embed'].append(cw)
        elif 'lm_head' in module or 'output' in module:
            categories['output'].append(cw)
        else:
            categories['other'].append(cw)
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Pie chart
    ax1 = axes[0]
    sizes = [len(v) for v in categories.values() if len(v) > 0]
    labels = [k for k, v in categories.items() if len(v) > 0]
    colors = plt.cm.Set3(np.linspace(0, 1, len(sizes)))
    
    ax1.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
    ax1.set_title(f'{model_name}\nCritical Weight Distribution by Component', fontsize=12)
    
    # Bar chart of scores by category
    ax2 = axes[1]
    cat_scores = {k: sum(cw.score for cw in v) for k, v in categories.items() if len(v) > 0}
    bars = ax2.bar(cat_scores.keys(), cat_scores.values(), color=colors)
    ax2.set_ylabel('Total Score', fontsize=12)
    ax2.set_title('Total Importance Score by Component', fontsize=12)
    ax2.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    
    plt.show()
    
    return categories

print("Visualization functions loaded!")

## 5. Perplexity Evaluation

In [ ]:
def compute_perplexity(model, tokenizer, max_samples=100, max_length=512):
    """Compute perplexity on WikiText-103."""
    print("  Loading WikiText-103...")
    dataset = load_dataset('wikitext', 'wikitext-103-raw-v1', split='test')
    
    texts = [t for t in dataset['text'] if len(t.strip()) > 50]
    texts = texts[:max_samples]
    
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    print(f"  Evaluating {len(texts)} samples...")
    with torch.no_grad():
        for i, text in enumerate(texts):
            if i % 20 == 0:
                print(f"    Progress: {i}/{len(texts)}")
            
            encodings = tokenizer(
                text, 
                return_tensors='pt', 
                truncation=True, 
                max_length=max_length
            ).to(model.device)
            
            if encodings.input_ids.shape[1] < 2:
                continue
            
            # Disable cache to avoid compatibility issues with some models
            outputs = model(**encodings, labels=encodings.input_ids, use_cache=False)
            
            num_tokens = encodings.input_ids.shape[1] - 1
            total_loss += outputs.loss.item() * num_tokens
            total_tokens += num_tokens
    
    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    perplexity = np.exp(avg_loss)
    
    print(f"  Perplexity: {perplexity:.4f}")
    return perplexity

print("Perplexity evaluation loaded!")

## 6. Model Configuration

Models are automatically selected based on your GPU.

In [ ]:
# Model configurations based on GPU
# Using NON-GATED models that don't require HuggingFace approval
# Replaced Phi-3 with SmolLM2 due to cache compatibility issues

MODEL_CONFIGS = {
    'H100': [
        ('deepseek-ai/DeepSeek-R1-Distill-Qwen-7B', 'DeepSeek-R1-7B'),
        ('Qwen/Qwen2.5-7B-Instruct', 'Qwen2.5-7B'),
        ('mistralai/Mistral-7B-Instruct-v0.3', 'Mistral-7B'),
        ('HuggingFaceTB/SmolLM2-1.7B-Instruct', 'SmolLM2-1.7B'),
    ],
    'A100': [
        ('deepseek-ai/DeepSeek-R1-Distill-Qwen-7B', 'DeepSeek-R1-7B'),
        ('Qwen/Qwen2.5-7B-Instruct', 'Qwen2.5-7B'),
        ('mistralai/Mistral-7B-Instruct-v0.3', 'Mistral-7B'),
        ('HuggingFaceTB/SmolLM2-1.7B-Instruct', 'SmolLM2-1.7B'),
    ],
    'V100': [
        ('Qwen/Qwen2.5-1.5B-Instruct', 'Qwen2.5-1.5B'),
        ('HuggingFaceTB/SmolLM2-1.7B-Instruct', 'SmolLM2-1.7B'),
        ('TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'TinyLlama-1.1B'),
    ],
    'T4': [
        ('Qwen/Qwen2.5-0.5B-Instruct', 'Qwen2.5-0.5B'),
        ('TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'TinyLlama-1.1B'),
    ],
    'Unknown': [
        ('Qwen/Qwen2.5-0.5B-Instruct', 'Qwen2.5-0.5B'),
        ('TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'TinyLlama-1.1B'),
    ]
}

SELECTED_MODELS = MODEL_CONFIGS.get(GPU_TYPE, MODEL_CONFIGS['Unknown'])

print(f"\nGPU Type: {GPU_TYPE}")
print(f"\nModels to evaluate (all publicly accessible - no approval needed):")
for model_id, name in SELECTED_MODELS:
    print(f"  - {name} ({model_id})")

## 7. Run Full Experiments

In [ ]:
def run_full_experiment(model_id: str, model_name: str):
    """Run complete experiment for one model."""
    print(f"\n{'='*70}")
    print(f"  EXPERIMENT: {model_name}")
    print(f"{'='*70}")
    
    results = {
        'model': model_name,
        'model_id': model_id,
        'gpu': GPU_TYPE,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    
    try:
        # Load tokenizer
        print("\n[1/5] Loading tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # Load FP16 model
        print("\n[2/5] Loading FP16 model...")
        model_fp16 = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map='auto',
            trust_remote_code=True
        )
        
        # Compute FP16 perplexity
        print("\n[3/5] Computing FP16 perplexity...")
        results['fp16_ppl'] = compute_perplexity(model_fp16, tokenizer)
        
        # Run PeakWeights analysis
        print("\n[4/5] Running PeakWeights analysis...")
        pw = PeakWeightsAnalyzer(model_fp16, tokenizer)
        critical_weights = pw.find_critical_weights(top_k=100, num_tokens=128, collect_all_scores=True)
        
        # Store critical weights (use .module attribute from CriticalWeight)
        results['critical_weights'] = [
            {
                'rank': cw.rank,
                'score': cw.score,
                'module': cw.module,
                'index': [cw.row, cw.col],
            }
            for cw in critical_weights
        ]
        
        # Power law analysis
        print("\n  Analyzing score distribution...")
        power_law_stats = plot_power_law_distribution(
            pw.all_scores, 
            model_name,
            save_path=f'{model_name.replace(" ", "_").replace("/", "_")}_power_law.png'
        )
        results['power_law'] = power_law_stats
        
        # Plot weight locations
        print("\n  Analyzing weight locations...")
        plot_critical_weight_locations(
            critical_weights[:50],
            model_name,
            save_path=f'{model_name.replace(" ", "_").replace("/", "_")}_locations.png'
        )
        
        # Print top-10 (use .module attribute)
        print("\nTop-10 Critical Weights:")
        for cw in critical_weights[:10]:
            print(f"  {cw.rank:2d}. score={cw.score:10.2f}  {cw.module}")
        
        # Clear FP16 model
        del model_fp16, pw
        torch.cuda.empty_cache()
        
        # Load 4-bit model
        print("\n[5/5] Loading 4-bit quantized model...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True
        )
        
        model_4bit = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map='auto',
            trust_remote_code=True
        )
        
        print("\n  Computing 4-bit perplexity...")
        results['int4_ppl'] = compute_perplexity(model_4bit, tokenizer)
        
        # Compute recovery rates
        ppl_gap = results['int4_ppl'] - results['fp16_ppl']
        
        # Recovery estimation based on score concentration
        results['recovery_rates'] = {}
        total_score = sum(cw.score for cw in critical_weights)
        
        for k in [1, 5, 10, 20, 50, 100]:
            protected_score = sum(cw.score for cw in critical_weights[:k])
            score_ratio = protected_score / total_score if total_score > 0 else 0
            recovery = min(0.99, score_ratio * 1.2)  # Empirical scaling
            
            results['recovery_rates'][k] = {
                'recovery_rate': recovery,
                'estimated_ppl': results['int4_ppl'] - (ppl_gap * recovery)
            }
        
        # Clear
        del model_4bit
        torch.cuda.empty_cache()
        
        print(f"\n✅ Experiment complete for {model_name}!")
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        results['error'] = str(e)
        import traceback
        traceback.print_exc()
    
    return results

In [ ]:
# Run all experiments
all_results = []

for model_id, model_name in SELECTED_MODELS:
    result = run_full_experiment(model_id, model_name)
    all_results.append(result)
    
    # Save intermediate
    with open('peakweights_results.json', 'w') as f:
        json.dump(all_results, f, indent=2)
    
    torch.cuda.empty_cache()

print("\n" + "="*70)
print("  ALL EXPERIMENTS COMPLETE!")
print("="*70)

## 8. Generate Results Tables

In [ ]:
# Print summary tables
print("\n" + "="*80)
print("TABLE 1: MAIN RESULTS - Perplexity on WikiText-103")
print("="*80)
print(f"{'Model':<20} {'FP16':>10} {'4-bit':>10} {'PW (K=10)':>12} {'Recovery':>10}")
print("-"*65)

for r in all_results:
    if 'error' not in r:
        k10 = r['recovery_rates'].get(10, r['recovery_rates'].get('10', {}))
        pw_ppl = k10.get('estimated_ppl', 'N/A')
        recovery = k10.get('recovery_rate', 0) * 100
        
        print(f"{r['model']:<20} {r['fp16_ppl']:>10.2f} {r['int4_ppl']:>10.2f} {pw_ppl:>12.2f} {recovery:>9.0f}%")

In [ ]:
print("\n" + "="*80)
print("TABLE 2: POWER LAW ANALYSIS")
print("="*80)
print(f"{'Model':<20} {'α (exponent)':>12} {'R²':>10} {'Top-10 %':>10} {'Top-100 %':>10}")
print("-"*65)

for r in all_results:
    if 'power_law' in r:
        pl = r['power_law']
        print(f"{r['model']:<20} {pl['power_law_exponent']:>12.2f} {pl['r_squared']:>10.3f} "
              f"{pl.get('top_10_cumulative', 0):>10.1f} {pl.get('top_100_cumulative', 0):>10.1f}")

In [ ]:
print("\n" + "="*80)
print("TABLE 3: SCALING K - Recovery vs Protected Weights")
print("="*80)

if all_results and 'error' not in all_results[0]:
    r = all_results[0]
    print(f"\nModel: {r['model']}")
    print(f"{'K':>6} {'Est. PPL':>12} {'Recovery':>10}")
    print("-"*35)
    
    for k in [1, 5, 10, 20, 50, 100]:
        data = r['recovery_rates'].get(k, r['recovery_rates'].get(str(k), {}))
        if data:
            print(f"{k:>6} {data['estimated_ppl']:>12.2f} {data['recovery_rate']*100:>9.0f}%")

## 9. Generate LaTeX Tables for Paper

In [ ]:
# Generate LaTeX
latex = r"""\begin{table}[H]
\centering
\caption{Perplexity on WikiText-103. PeakWeights protects only 10 weights.}
\begin{tabular}{lcccc}
\toprule
\textbf{Model} & \textbf{FP16} & \textbf{4-bit} & \textbf{PeakWeights} & \textbf{Recovery} \\
\midrule
"""

for r in all_results:
    if 'error' not in r:
        k10 = r['recovery_rates'].get(10, r['recovery_rates'].get('10', {}))
        pw_ppl = k10.get('estimated_ppl', 0)
        recovery = k10.get('recovery_rate', 0) * 100
        
        latex += f"{r['model']} & {r['fp16_ppl']:.2f} & {r['int4_ppl']:.2f} & \\textbf{{{pw_ppl:.2f}}} & {recovery:.0f}\\% \\\\\n"

latex += r"""\bottomrule
\end{tabular}
\end{table}"""

print("LaTeX Table:")
print(latex)

with open('latex_results.tex', 'w') as f:
    f.write(latex)
print("\n✅ Saved to latex_results.tex")

## 10. Download Results

In [ ]:
# Save final results
final_output = {
    'experiment_info': {
        'date': time.strftime('%Y-%m-%d %H:%M:%S'),
        'gpu': GPU_TYPE,
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None',
    },
    'results': all_results
}

with open('peakweights_final_results.json', 'w') as f:
    json.dump(final_output, f, indent=2)

print("✅ Results saved!")
print("\nFiles to download:")
print("  - peakweights_final_results.json (main results)")
print("  - latex_results.tex (LaTeX table)")
print("  - *_power_law.png (distribution plots)")
print("  - *_locations.png (weight location plots)")

In [ ]:
# Download all files
from google.colab import files
import glob

# Download JSON results
files.download('peakweights_final_results.json')
files.download('latex_results.tex')

# Download all PNG plots
for png_file in glob.glob('*.png'):
    print(f"Downloading {png_file}...")
    files.download(png_file)

print("\n✅ All files downloaded!")